# RIM Experiment Analysis

Loads one experiment run and plots the key signals.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from rim_teleop.data import ExperimentRegistry, load_streams

DATA_DIR = "/home/athena/csirois/data/franka/rim"

registry = ExperimentRegistry(DATA_DIR)

# --- Select run ---
run = registry.latest()                         # most recent
# run = registry.get("run_20260521_160401")     # specific run

print(f"Run: {run.name}  ({run.created_at})")
print(f"rim_enabled={run.config['interface']['rim_enabled']}  "
      f"force_feedback={run.config['interface']['force_feedback']}  "
      f"K={run.config['interface']['stiffness']}  "
      f"D={run.config['interface']['damping']}")

In [ ]:
streams = load_streams(run)
print("Available streams:", list(streams.keys()))
for name, df in streams.items():
    print(f"  {name}: {len(df)} rows | {list(df.columns)}")

In [ ]:
# Normalise timestamps to t=0
t0 = min(df["ts"].min() for df in streams.values())
for df in streams.values():
    df["t"] = df["ts"] - t0

## Position tracking — TCP, tool tip, RIM proxy, sent target

In [ ]:
if "tracking" in streams:
    tr = streams["tracking"]
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(tr["t"], tr["tcp"],      label="TCP (robot EE)",   lw=1.0)
    ax.plot(tr["t"], tr["tool_tip"], label="Tool tip (Pinocchio)", lw=1.0, ls="--")
    ax.plot(tr["t"], tr["rim"],      label="RIM proxy",         lw=1.0, ls=":")
    ax.plot(tr["t"], tr["target"],   label="Sent target (TCP)", lw=0.8, alpha=0.6)
    ax.set(xlabel="Time (s)", ylabel="Position along axis (m)", title="Position tracking")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## Haptic — leader position and force command

In [ ]:
if "haptic" in streams:
    hap = streams["haptic"]
    fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

    axes[0].plot(hap["t"], hap["leader_pos"], lw=0.6, label="Leader position")
    axes[0].set(ylabel="Position (m)", title="Haptic device")
    axes[0].legend()

    axes[1].plot(hap["t"], hap["leader_vel"], lw=0.6, label="Leader velocity")
    axes[1].set(ylabel="Velocity (m/s)")
    axes[1].legend()

    axes[2].plot(hap["t"], hap["force_cmd"],          lw=0.8, label="Force cmd (sent)")
    if "rim_interface_force" in hap.columns:
        axes[2].plot(hap["t"], hap["rim_interface_force"], lw=0.6, alpha=0.7, label="RIM interface force")
    if "robot_force" in hap.columns:
        axes[2].plot(hap["t"], hap["robot_force"],         lw=0.6, alpha=0.7, label="Robot force (filtered)")
    axes[2].set(xlabel="Time (s)", ylabel="Force (N)")
    axes[2].legend()

    for ax in axes:
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## RIM model parameters

In [ ]:
if "rim_model" in streams:
    rm = streams["rim_model"]
    fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)

    axes[0].plot(rm["t"], rm["M_eff"], lw=1.0)
    axes[0].set(ylabel="M_eff (kg)", title="RIM model parameters")

    axes[1].plot(rm["t"], rm["f_eff"], lw=1.0, label="f_eff")
    axes[1].plot(rm["t"], rm["z_i"],   lw=1.0, label="z_i", alpha=0.7)
    axes[1].set(ylabel="Force terms (N)")
    axes[1].legend()

    axes[2].plot(rm["t"], rm["x"], lw=1.0, label="x_rim (at control rate)")
    axes[2].plot(rm["t"], rm["v"], lw=1.0, label="v_rim", alpha=0.7)
    axes[2].set(xlabel="Time (s)", ylabel="State")
    axes[2].legend()

    for ax in axes:
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## Loop rate metrics

In [ ]:
if "metrics" in streams:
    met = streams["metrics"]
    print("Loop rate summary (last snapshot):")
    last = met.iloc[-1]
    for loop in ["haptic", "rim", "control"]:
        hz_col = f"{loop}.measured_hz"
        p95_col = f"{loop}.p95_dt_ms"
        if hz_col in last:
            print(f"  {loop:8s}: {last[hz_col]:.1f} Hz  p95={last[p95_col]:.2f} ms")

## Force spectrum — instability diagnosis

FFT of the haptic force command to identify oscillation frequency.

In [ ]:
import numpy as np

if "haptic" in streams:
    hap = streams["haptic"]
    # Use only a contiguous segment at nominally 1 kHz
    force = hap["force_cmd"].values
    ts = hap["ts"].values
    dt_median = float(np.median(np.diff(ts)))
    fs = 1.0 / dt_median

    freqs = np.fft.rfftfreq(len(force), d=dt_median)
    spectrum = np.abs(np.fft.rfft(force - force.mean()))

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.semilogy(freqs, spectrum, lw=0.6)
    ax.set(xlabel="Frequency (Hz)", ylabel="Amplitude", title="Force command spectrum")
    ax.set_xlim(0, min(fs / 2, 200))
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    top_freqs = freqs[np.argsort(spectrum)[-5:][::-1]]
    print(f"Measured haptic rate: {fs:.0f} Hz")
    print(f"Top 5 spectral peaks: {top_freqs.round(1)} Hz")